# 3 – Grafo por ejes + Enrichment + KEGG overlay

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import networkx as nx

from intergate.config import CFG
from intergate.utils import set_seed, set_all_seeds, cleanup_memory
from intergate.data import (
    load_expression_and_metadata, prepare_genes, encode_labels,
    cohort_split, scale_features, apply_connected_only,
    make_dataloaders, build_regulator_features, ExpressionDataset,
)
from intergate.graph import build_backbone
from intergate.graph_cache import get_or_build_backbone, get_or_build_Xh
from intergate.losses import compute_metrics_full, compute_class_weights_balanced
from intergate.training import predict_proba_xh_mode
from intergate.pruning import export_pruned_graph
from intergate.ablation import (
    AblationConfig, load_bundle, build_pruned_model_from_bundle,
    register_runtime, get_full_gene_matrix_and_genes,
    run_single_seed, run_ablation,
)



from intergate.axes import AXES
from intergate.axes import draw_graph_colored_by_axes_from_bundle
from intergate.visualization import (
    draw_global_graph_from_bundle,
    make_dotplot, make_barplot, get_node_connections_from_bundle
)
from intergate.enrichment import (
    run_gprofiler, run_enrichr_battery,
    build_symbol_entrez_maps, plot_kegg_map_overlay,
)
set_seed(CFG.SEED)
DEVICE = CFG.DEVICE
print("DEVICE:", DEVICE)


## 1) Data pipeline

In [ ]:
X_df, y_str, cohort = load_expression_and_metadata(
    CFG.EXPR_CSV, CFG.META_CSV,
    sample_col=CFG.SAMPLE_COL, label_col=CFG.LABEL_COL, cohort_col=CFG.COHORT_COL,
)
X_df_kegg, genes_kegg = prepare_genes(X_df)
y, classes, label_map = encode_labels(y_str)
n_classes = len(classes)


In [ ]:
edge_index, edge_weight, edge_type = get_or_build_backbone(
    genes_kegg, cache_dir=CFG.PIPELINE_CACHE_DIR,
    force_rebuild=False, use_omnipath=CFG.USE_OMNIPATH, use_huri=CFG.USE_HURI,
)
print(f"Backbone: {edge_index.shape[1]} aristas, {len(genes_kegg)} genes")


In [ ]:
train_idx, val_idx, test_idx = cohort_split(
    cohort, y, train_cohort_frac=0.80, val_size=CFG.VAL_SIZE, seed=CFG.SEED,
)
Xs_gene = scale_features(X_df_kegg, train_idx, mode=CFG.SCALE_MODE, use_quantile=CFG.USE_QUANTILE)

if CFG.CONNECTED_ONLY:
    Xs_gene, edge_index, edge_weight, edge_type, genes_kegg = apply_connected_only(
        Xs_gene, edge_index, edge_weight, edge_type, genes_kegg,
    )


In [ ]:
Xs_graph, graph_feat_names = get_or_build_Xh(
    Xs_gene, genes_kegg, edge_index,
    cache_dir=CFG.PIPELINE_CACHE_DIR,
    stats=CFG.REG_STATS, min_targets=CFG.REG_MIN_GENES,
    max_regulators=CFG.REG_MAX_REGULATORS,
)


In [ ]:
dl_tr, dl_va, dl_te = make_dataloaders(
    Xs_gene, Xs_graph, y, train_idx, val_idx, test_idx, batch_size=CFG.BATCH_SIZE,
)


In [ ]:
edge_index_t = torch.as_tensor(edge_index, dtype=torch.long, device=DEVICE)
edge_weight_t = torch.as_tensor(edge_weight, dtype=torch.float32, device=DEVICE)
edge_type_t = torch.as_tensor(edge_type, dtype=torch.long, device=DEVICE)

register_runtime(
    Xs_gene=Xs_gene, genes_kegg=genes_kegg, X_h=Xs_graph,
    y=y, n_classes=n_classes, label_map=label_map,
    train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
    edge_index_t=edge_index_t, edge_weight_t=edge_weight_t, edge_type_t=edge_type_t,
)


## 2) Cargar bundle y extraer genes del subgrafo

In [ ]:
BUNDLE_DIR = os.path.join(CFG.ARTIFACTS_ROOT, "FULL", "seed_1234")
bundle = load_bundle(BUNDLE_DIR)
g = bundle["graph"]

nodes_used = np.asarray(g["nodes_used_full"], dtype=np.int64)
X_full, genes_full = get_full_gene_matrix_and_genes()
genes_model = [str(genes_full[i]) for i in nodes_used]

# Genes del subgrafo (con grado > 0)
edge_index_p = g.get("edge_index_compact", g.get("edge_index_comp")).cpu()
node_ids = torch.unique(edge_index_p.reshape(-1)).cpu().numpy()
genes_subgraph = sorted(set(genes_model[i] for i in node_ids))
print(f"Genes en subgrafo: {len(genes_subgraph)}")


In [ ]:


AXIS_PRIORITY = list(AXES.keys())
TOPK_LABELS_PER_AXIS = 3
SHOW_GIANT_COMPONENT = False  # ← como en el notebook TNBC

# ── Genes: misma fuente que el notebook TNBC ──
if "genes_comp" in bundle:
    _genes = bundle["genes_comp"]
else:
    _genes = bundle["graph"]["genes_comp"]
if isinstance(_genes, np.ndarray):
    _genes = _genes.tolist()
_genes = [str(g) for g in _genes]

# ── Edge index ──
if "edge_index_comp" in bundle:
    ei = bundle["edge_index_comp"]
else:
    ei = g.get("edge_index_compact", g.get("edge_index_comp"))
ei = ei.cpu().numpy() if hasattr(ei, 'cpu') else np.asarray(ei)

# ── Construir G ──
G = nx.DiGraph()
G.add_nodes_from(_genes)
for k in range(ei.shape[1]):
    u, v = _genes[int(ei[0, k])], _genes[int(ei[1, k])]
    G.add_edge(u, v)

if SHOW_GIANT_COMPONENT:
    gcc = max(nx.weakly_connected_components(G), key=len)
    G = G.subgraph(gcc).copy()

# ── Asignación de ejes ──
gene_to_axes = {}
for axis, gset in AXES.items():
    for gene in gset:
        gene_to_axes.setdefault(gene, []).append(axis)

def assign_axis(gene):
    axes_here = gene_to_axes.get(gene, [])
    if not axes_here:
        return "Residual"
    for ax in AXIS_PRIORITY:
        if ax in axes_here:
            return ax
    return axes_here[0]

node_axis = {n: assign_axis(n) for n in G.nodes()}
deg = dict(G.degree())

# ── Tabla ──
rows = []
for ax in AXIS_PRIORITY:
    genes_axis = [n for n in G.nodes() if node_axis[n] == ax]
    genes_axis_sorted = sorted(genes_axis, key=lambda x: deg[x], reverse=True)
    rep_hubs = genes_axis_sorted[:TOPK_LABELS_PER_AXIS]
    rows.append({
        "Visible module": ax,
        "n nodes": len(genes_axis),
        "Representative hubs": ", ".join(f"\\texttt{{{g}}}" for g in rep_hubs),
        "All visible genes": ", ".join(f"\\texttt{{{g}}}" for g in genes_axis_sorted),
    })

df_sup = pd.DataFrame(rows)
display(df_sup)
df_latex = df_sup[["Visible module", "n nodes", "Representative hubs", "All visible genes"]].copy()
print(df_latex.to_latex(index=False, escape=False))

In [ ]:
# ── Extraer genes residuales (no cubiertos por ningún eje) ──
axis_gene_union = set()
for _, gset in AXES.items():
    axis_gene_union.update(gset)

residual_genes = [g for g in genes_subgraph if g not in axis_gene_union]
axis_genes_in_subgraph = [g for g in genes_subgraph if g in axis_gene_union]

print(f"Genes en ejes (presentes en subgrafo): {len(axis_genes_in_subgraph)}")
print(f"Genes residuales: {len(residual_genes)}")


## 3) Grafo coloreado por ejes

"#2F2F2F" — gris carbón
"#1B1B3A" — azul noche (suficientemente distinto del azul HuRI)
"#3B2F2F" — marrón oscuro
"#2D4A3E" — verde bosque oscuro
"#4A3728" — tierra/café

Pastel:

"#F4A6C1" — rosa pastel
"#A8D8EA" — azul cielo
"#C3B1E1" — lila
"#F9E79F" — amarillo suave
"#ABEBC6" — menta
"#F5CBA7" — melocotón
"#D5DBDB" — gris perla

plt.cm.plasma — similar a magma pero el extremo bajo es más morado/violeta, no tan negro. Mejor pero no ideal.
plt.cm.inferno — mismo problema que magma en el extremo bajo.
plt.cm.YlOrRd — amarillo → naranja → rojo. Siempre visible, nunca negro. Mi recomendación si quieres contraste con las aristas azul/verde/rojo y etiquetas oscuras.
plt.cm.YlGnBu — amarillo → verde → azul. Bonita pero puede confundirse con aristas verdes/azules.
plt.cm.RdPu — rosa → morado. No compite con ningún color de arista.
plt.cm.Oranges — escala de naranjas. Limpia, siempre legible, no compite con aristas.



In [ ]:
draw_global_graph_from_bundle(
    bundle,
    title="",
    layout="kamada_kawai", #axis_modules,kamada_kawai
    seed=42,
    figsize=(18, 15), facecolor="#0d1117",
    
    #nodos
    node_color_mode="score", #"score" | "constant" | "degree"
    node_size_mode="degree", 
    node_cmap=plt.cm.plasma,
    node_size_base=80.0, 
    node_size_scale=500.0,
    node_alpha=0.92, 
    node_edgecolor="#000000", 
    node_linewidth=0.6,
    
    #labels
    top_labels=10, 
    force_label_genes=["ESR1", "ERBB2", "TP53", "EGFR", "ABL1"],
    label_fontsize=12, 
    label_fontcolor="#3B2F2F",#colores
    label_fontweight="bold", 
    label_bbox_alpha=1.0,
    
    #arista
    edge_color_mode="etype", #"constant" si las quiero todas en gris oscuro
    #edge_color_constant="#444444",#cambiar color de las arista a gris oscuro y para un nodo que digamos, sacar sus conexiones y tipo de arista inhinibico, activación huri PPI. tanto las que llegan como las que salen. 
    edge_width_mode="weight", 
    edge_width_min=1.2, 
    edge_width_max=4.0, 
    edge_alpha=0.8,
    
    #tipo flecha
    arrows=True, 
    arrowstyle="-|>", 
    arrowsize=14,
    connectionstyle="arc3,rad=0.08",
   
    #barras
    show_colorbar_nodes=False, 
    show_colorbar_edges=False,
    annotate_stats=False,
    save_path="./figures_axes_refined/Grafo_final.png", 
    dpi=300,
    # ★ genes destacados
    highlight_genes=["ESR1", "ERBB2", "TP53", "EGFR", "ABL1"],
    highlight_size=200.0,
    highlight_color="#CC5500",
    highlight_offset_frac=0.001
)

In [ ]:
df_node = get_node_connections_from_bundle(bundle, "ESR1")
display(df_node)

## 4) Enrichment (gProfiler + Enrichr)

In [ ]:
# ── Enrichment con genes RESIDUALES (no cubiertos por ejes) ──
gene_list = [g.strip().upper() for g in residual_genes]
gene_symbols_125 = gene_list  # alias para compatibilidad
os.makedirs("./Results", exist_ok=True)

print(f"Enriquecimiento sobre {len(gene_list)} genes residuales")

# gProfiler
df_gp = run_gprofiler(gene_list, save_csv="./Results/enrich_gprofiler_GO_REAC_KEGG.csv")
if not df_gp.empty:
    display(df_gp.head(20))

# Enrichr battery
results_enrichr = run_enrichr_battery(gene_list, out_prefix="enrich_keep", save_dir="./Results")
for tag, df in results_enrichr.items():
    if not df.empty:
        print(f"[{tag}] {len(df)} terms")


## 5) KEGG pathway overlay

In [ ]:
symbol_to_entrez, entrez_to_symbol = build_symbol_entrez_maps(residual_genes)
unmapped = [g for g in gene_list if g not in symbol_to_entrez]
print(f"Unmapped: {len(unmapped)}")

for pathway in ["hsa05224", "hsa04110", "hsa05200", "hsa04914"]:
    try:
        plot_kegg_map_overlay(
            pathway, selected_genes=gene_symbols_125,
            symbol_to_entrez=symbol_to_entrez,
            entrez_to_symbol=entrez_to_symbol,
        )
    except Exception as e:
        print(f"[WARN] {pathway}: {e}")


## 5) Dotplot Barplot


In [ ]:
# ============================================================
# Construir df_plot unificado (gProfiler + Enrichr)
# ============================================================
TOP_N = 5  # términos por fuente

frames = []

# — gProfiler —
if not df_gp.empty:
    tmp = df_gp[df_gp["significant"]].copy()
    tmp = tmp.rename(columns={"name": "label", "source": "source"})
    tmp["neg_log10_p"] = -np.log10(tmp["p_value"].clip(lower=1e-300))
    tmp["gene_ratio"]  = tmp["intersection_size"] / tmp["query_size"]
    frames.append(tmp[["label", "source", "gene_ratio", "neg_log10_p", "intersection_size"]])

# — Enrichr battery —
for tag, df_e in results_enrichr.items():
    if df_e.empty:
        continue
    tmp = df_e[df_e["Adjusted P-value"] < 0.05].copy()
    tmp["label"]  = tmp["Term"]
    tmp["source"] = tag
    tmp["neg_log10_p"]      = -np.log10(tmp["Adjusted P-value"].clip(lower=1e-300))
    tmp["intersection_size"] = tmp["Overlap"].str.split("/").str[0].astype(int)
    query_n                  = tmp["Overlap"].str.split("/").str[1].astype(int)
    tmp["gene_ratio"]        = tmp["intersection_size"] / query_n
    frames.append(tmp[["label", "source", "gene_ratio", "neg_log10_p", "intersection_size"]])

df_all = pd.concat(frames, ignore_index=True)

df_plot = (
    df_all
    .sort_values("neg_log10_p", ascending=False)
    .groupby("source", sort=False).head(TOP_N)
    .reset_index(drop=True)
    .sort_values("neg_log10_p")
)

# ============================================================
# Colores y rutas
# ============================================================
SOURCE_COLORS = {
    "GO:BP": "#E69F00",  "GO:MF": "#56B4E9",  "GO:CC": "#009E73",
    "REAC":  "#CC79A7",  "KEGG":  "#D55E00",
    # aliases Enrichr
    "GO_BP": "#E69F00",  "GO_MF": "#56B4E9",  "GO_CC": "#009E73",
    "REACTOME": "#CC79A7",
}




In [ ]:
df_plot = df_plot.loc[df_plot['source'].isin(['REACTOME','GO_BP','KEGG'])]
df_plot

In [ ]:
OUT_DOTPLOT = "./Results/dotplot_enrichment.svg"
OUT_BARPLOT = "./Results/barplot_enrichment.svg"
# ============================================================
# Dotplot
# ============================================================
make_dotplot(
    df=df_plot,
    out_path=OUT_DOTPLOT,
    source_colors=SOURCE_COLORS,
    label_col="label",
    source_col="source",
    gene_ratio_col="gene_ratio",
    pval_col="neg_log10_p",
    size_col="intersection_size",
    figsize=(12, None),
    dpi=300,
    facecolor="white",
    axes_facecolor="white",
    clean_labels=True,
    title="",
    xlabel="Gene ratio",
    colorbar_label="−log₁₀(adj. p-value)",
    size_legend_title="Genes overlap",
    source_legend_title="Source",
    cmap_colors=("#9BD7F4", "#6C8EBF", "#7B61A8", "#D55E00"),
    grid_color="#D9D9D9",
    spine_color="#B0B0B0",
    text_color="black",
    edgecolor_points="white",
    fontsize_label=14,       # colorbar label
    fontsize_tick=14,         # colorbar ticks + x-axis ticks
    fontsize_ytick=14,        # y-axis pathway labels
    fontsize_legend=10,       # ambas leyendas
    fontsize_xlabel=16,       # xlabel
    fontsize_title=16,        # título
    panel_label="B",
    panel_label_x=-0.02,
    panel_label_y=1.0,
    panel_label_fontsize=18,
    panel_label_bold=True,
    save=True,
)

make_barplot(
    df=df_plot,
    out_path=OUT_BARPLOT,
    source_colors=SOURCE_COLORS,
    label_col="label",
    source_col="source",
    pval_col="neg_log10_p",
    size_col="intersection_size",
    figsize=(12, None),
    dpi=300,
    facecolor="white",
    axes_facecolor="white",
    fdr_cutoff=0.05,
    clean_labels=True,
    title="",
    xlabel="−log₁₀(adjusted p-value)",
    source_legend_title="Source",
    show_counts=True,
    grid_color="#D9D9D9",
    spine_color="#B0B0B0",
    text_color="black",
    fdr_line_color="#D55E00",
    fontsize_counts=10,      # "n=X" junto a barras
    fontsize_ytick=14,        # y-axis pathway labels
    fontsize_xtick=14,        # x-axis ticks
    fontsize_legend=10,       # leyenda
    fontsize_xlabel=16,       # xlabel
    fontsize_title=16,        # título
    panel_label="A",
    panel_label_x=-0.02,
    panel_label_y=1.0,
    panel_label_fontsize=18,
    panel_label_bold=True,
    save=True,
)

## 5) Grafo residuales

In [ ]:


fig, G, df_membership = draw_graph_colored_by_axes_from_bundle(
    bundle,

    # ── Layout ─────────────────────────────────────────
    directed=True,
    giant_component=False,
    layout="axis_modules",
    module_radius=5.0,           # NO más de ~8-10
    module_jitter=8.80,          # NO más de ~1.5
    spring_iter=600,             # 400-800 es suficiente
    seed=42,

    # ── Nodos ──────────────────────────────────────────
    base_node_size=120,
    degree_size_scale=38,
    residual_node_scale=0.90,
    node_alpha_axis=0.96,
    node_alpha_residual=0.72,
    node_edgewidth=0.9,
    node_edgewidth_multi=2.0,

    # ── Aristas — MÁS VISIBLES ────────────────────────
    edge_alpha_intra=0.85,          # ↑ intra-módulo
    edge_alpha_cross=0.85,          # ↑ entre módulos distintos
    edge_alpha_to_residual=0.8,    # ↑↑ era 0.06, ahora se ven
    edge_alpha_residual=0.68,       # ↑ residual-residual
    edge_width_min=0.5,
    edge_width_max=2.0,

    # ── Etiquetas — TODOS los de eje + top residuales ─
    label_mode="topk_per_axis",
    topk_per_axis=99,           # ← poner un número alto = todos los del eje
    topk_residual=15,           # ← más residuales visibles también
    always_label_multi=True,
    label_fontsize=7,           # ↓ un poco más pequeño para que no se solapen

    # ── Figura ─────────────────────────────────────────
    figsize=(18, 14),
    dpi=300,
    title="",#"Subgraph colored by biological axis",
    title_fontsize=17,

    # ── Guardar ────────────────────────────────────────
    save_path="./figures_axes_refined/fig_axis_graph_colored_by_axes.png",
    show=True,
)

display(df_membership.head(10))


In [ ]:

# Genes del bundle
genes_in_graph = set(df_membership["gene"])

# Todos los genes de eje
all_axis_genes = {g for glist in AXES.values() for g in glist}

missing = all_axis_genes - genes_in_graph
present = all_axis_genes & genes_in_graph
isolated = [g for g in present if df_membership.loc[df_membership["gene"] == g, "degree"].values[0] == 0]

print(f"Genes de eje en grafo:  {len(present)}/{len(all_axis_genes)}")
print(f"Ausentes del subgrafo:  {len(missing)} → {sorted(missing)}")
print(f"Presentes pero grado 0: {len(isolated)} → {sorted(isolated)}")

In [ ]:
draw_global_graph_from_bundle(
    bundle,
    title="",
    layout="kamada_kawai",
    seed=42,
    figsize=(18, 15), facecolor="#0d1117",

    # ── nodos por eje biológico ────────────────────────
    node_color_mode="axis",          # ★ NUEVO
    node_size_mode="degree",
    node_size_base=80.0,
    node_size_scale=500.0,
    node_alpha=0.92,
    node_edgecolor="#000000",
    node_linewidth=0.6,

    # ── labels ─────────────────────────────────────────
    top_labels=30,
    force_label_genes=["ESR1", "ERBB2", "TP53", "EGFR", "ABL1"],
    label_fontsize=12,
    label_fontcolor="#000000",
    label_fontweight="bold",
    label_bbox_alpha=1.0,

    # ── aristas por eje biológico ──────────────────────
    edge_color_mode="axis",          # ★ NUEVO
    edge_width_mode="weight",
    edge_width_min=1.2,
    edge_width_max=4.0,
    edge_alpha=0.8,

    # ── flechas ────────────────────────────────────────
    arrows=True,
    arrowstyle="-|>",
    arrowsize=14,
    connectionstyle="arc3,rad=0.08",

    # ── extras ─────────────────────────────────────────
    show_colorbar_nodes=False,
    show_colorbar_edges=False,
    annotate_stats=False,
    save_path="./figures_axes_refined/fig_axis_graph_colored_by_axes.png",
    dpi=300,

    highlight_genes=["ESR1", "ERBB2", "TP53", "EGFR", "ABL1"],
    highlight_size=200.0,
    highlight_color="#CC5500",
    highlight_offset_frac=0.001,
)

# Fin